In [32]:
#!/usr/bin/env python3
import os
import sys
import pandas as pd
from tqdm import tqdm
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import numpy as np
from goatools.obo_parser import GODag
from goatools.gosubdag.gosubdag import GoSubDag
from goatools.mapslim import mapslim

In [33]:
MODULE_CORR_FILE = "../data/analysis/WGCNA_130/gene_module_correlations.csv"
ANNOTATION_FILE  = "../data/phaglo1_mapping/annotation/gene_functions.tsv"
OUTDIR           = "../data/analysis/WGCNA_130/enrichment"
# If out dir does not exist, create it
if not os.path.exists(OUTDIR):
    os.makedirs(OUTDIR)

# GO files
GO_OBO           = "../data/annotation/functional_eggnog/go-basic.obo"
GOSLIM_OBO       = "../data/annotation/functional_eggnog/goslim_generic.obo"

ANNOT_COLUMNS    = ["KEGG_def", "KEGG_pathway", "KOG_desc", "CAZy_desc"]  # GO handled separately
MIN_SIZE         = 5
QVAL_CUTOFF      = 0.10
EFFECT_CUTOFF    = 0.55

## Load data

In [34]:
print("Loading gene–module correlations…")
corr_df = pd.read_csv(MODULE_CORR_FILE)

print("Loading gene functions…")
ann = pd.read_csv(ANNOTATION_FILE, sep="\t")

Loading gene–module correlations…
Loading gene functions…


In [35]:
# Helper function to explode annotations with multiple categories per gene
def explode_annotations(df, col):
    sub = df[["gene_id", col]].dropna().copy()
    sub = sub[sub[col] != ""]
    sub[col] = sub[col].astype(str).str.split(";")
    sub = sub.explode(col)
    sub[col] = sub[col].str.strip()
    sub = sub[sub[col] != ""]
    sub.columns = ["gene_id", "annotation"]
    return sub

# Customize the MWU function to use 'gene_id' as the identifier column
def qvals_bh(pvals: np.array):
    return multipletests(pvals, method="fdr_bh")[1]

def enrich_mwu(ordered_list, annotation: pd.DataFrame,
               min_size=5, alternative="two-sided", print_progress=0):
    """
    Check if some annotation information occurs at the top/bottom of an ordered list of identifiers.

    :param ordered_list: List of KEGG KO, Module, transcript ids, etc. associated with a WGCNA module, ordered by some metric (e.g. moduleMembership).
    :param annotation: columns: "KEGG_ko", "annotation", "KEGG_Module", etc.
    :param min_size: the minimum occurrence of values in annotation category (that are also in ordered list)
        that we need in order to actually run the MWU test.
        I. e. smaller gene sets are skipped.
    :param alternative: passed to `mannwhitneyu`.
    :param print_progress: print a progress message every 'n' tests.
    :return: A data frame with columns:
        - annotation: the annotation category that was tested
        - p_val: the two-sided p-value of the MWU test:
          the more a term is located at the top/bottom, the lower the p-value.
        - q_val: the BH-adjusted p-value.
        - U: the MWU test-statistic.
        - size: the size of the overlap between the number of terms belonging to a annotation category and all terms in the list
        - estimate: the probability that a random term of the given category
        is ranked higher than another random term of the ordered list.
        - direction: can take three values: "top"/"mid"/"bot", if the estimate is bigger/equal/smaller than 0.5.
          For example, "top" means a term is located closer to the top of the given list.
    """
    # Get the indices of all identifiers in the ordered list.
    all_indices = list(range(len(ordered_list)))

    # Group the annotation datas by their categories.
    groups = annotation.groupby("annotation")
    n_groups = len(groups)
    results = []
    
    # Loop over each pathway.
    for i, (annotation, group) in enumerate(groups):
        # Print a progress message if print_progress is set and i is a multiple of print_progress
        if print_progress and i % print_progress == 0:
            print("{:6d} / {:6d} ({:05.2f}%)".format(i, n_groups, i/n_groups * 100))

        # Get the gene ids of the current pathway.
        group_gene_ids = group.gene_id.values
        
        # Find the indices of the transcript ids in the ordered list that are also in the current pathway.
        indices_of_annotation = np.flatnonzero(np.in1d(ordered_list, group_gene_ids))

        # Only perform the MWU test if the number of overlapping genes is at least min_size
        if len(indices_of_annotation) >= min_size:
            # Perform the Mann-Whitney U test between the indices of all genes in the ordered list and
            # the indices of the genes in the current GO category
            u2, p_val = mannwhitneyu(x=all_indices, y=indices_of_annotation, alternative=alternative)
            results.append({"annotation": annotation, "p_val": p_val, "U": u2, "size": len(indices_of_annotation)})

    # Record the results in a dictionary
    df = pd.DataFrame.from_records(results)
    
    print(f"Number of significant results: {df[df['p_val'] < 0.05].shape[0]}")
    # Add the corrected p-values
    df.insert(column="q_val", value=qvals_bh(df["p_val"]), loc=2)
    # Calculate the estimate and direction
    df["estimate"] = df["U"]/(df["size"] * len(all_indices))
    df["direction"] = df["estimate"].apply(lambda e: "mid" if e == 0.5 else ("top" if e > 0.5 else "bot"))
    df["_extremity"] = 0.5 - abs(0.5 - df["estimate"])
    df.sort_values(by=["p_val", "_extremity"], inplace=True)
    df.drop(columns="_extremity", inplace=True)
    return df

## Run MWU enrichment per module

In [36]:
for annot_col in ANNOT_COLUMNS:
    if annot_col not in ann.columns:
        print(f"Column {annot_col} not found in gene_functions.tsv, skipping.")
        continue

    print(f"\n=== Running enrichment for {annot_col} ===")

    annotations = explode_annotations(ann, annot_col)
    print(f"Annotated genes for {annot_col}: {annotations.gene_id.nunique()} / {ann.gene_id.nunique()}")

    all_results = []

    for module in tqdm(corr_df['module'].unique(), desc=f"Modules ({annot_col})"):
        sub = corr_df[corr_df['module'] == module].copy()
        sub = sub.sort_values(by="correlation", ascending=False)

        res = enrich_mwu(
            ordered_list=sub['gene_id'].values,
            annotation=annotations,
            min_size=MIN_SIZE,
            alternative="two-sided"
        )

        res['Module'] = module
        res = res[['Module','annotation','estimate','p_val','q_val','U','size','direction']]
        
        # Only keep annotations that are top enriched
        res = res[res['direction'] == 'top']

        # Save per-module results
        module_out = os.path.join(OUTDIR, f"{module}_{annot_col}_MWU.csv")
        res.to_csv(module_out, index=False)

        sig = res[(res['q_val'] < QVAL_CUTOFF) & (res['estimate'] > EFFECT_CUTOFF)]
        if not sig.empty:
            all_results.append(sig)

    # Combine significant results
    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        out_file = os.path.join(OUTDIR, f"MWU_significant_{annot_col}.csv")
        combined.to_csv(out_file, index=False)
        print(f"Saved combined significant {annot_col} results to {out_file}")
    else:
        print(f"No significant enrichments found for {annot_col}")


=== Running enrichment for KEGG_def ===
Annotated genes for KEGG_def: 2393 / 32196


Modules (KEGG_def):  33%|███▎      | 2/6 [00:00<00:00,  5.00it/s]

Number of significant results: 11
Number of significant results: 5


Modules (KEGG_def):  50%|█████     | 3/6 [00:00<00:00,  5.07it/s]

Number of significant results: 6


...

Number of significant results: 1


Modules (CAZy_desc):  83%|████████▎ | 5/6 [00:01<00:00,  4.35it/s]

Number of significant results: 1


Modules (CAZy_desc): 100%|██████████| 6/6 [00:01<00:00,  4.48it/s]

Number of significant results: 0
Saved combined significant CAZy_desc results to ../data/analysis/WGCNA_130/enrichment/MWU_significant_CAZy_desc.csv


In [37]:
print("\n=== Running GO enrichment (raw + slim) ===")

# Explode raw GO
go_annot = explode_annotations(ann, "GO")
print(f"Annotated genes for GO: {go_annot['gene_id'].nunique()} / {ann['gene_id'].nunique()}")

# Load DAGs
go_dag = GODag(GO_OBO, optional_attrs=['relationship'])
goslim_dag = GODag(GOSLIM_OBO)

# Map GO → name (for both raw + slim terms)
all_go_terms = set(go_annot['annotation'].unique()) | set(goslim_dag.keys())
go_id_to_name = {go: go_dag[go].name for go in all_go_terms if go in go_dag}

# Function to map full GO terms to slim terms
def map_to_goslim(go_ids):
    slim_terms = set()
    for go in go_ids:
        if go not in go_dag:
            continue
        try:
            mapping = mapslim(go, go_dag, goslim_dag)
            for _, slim_list in mapping.items():
                slim_terms.update(slim_list)
        except Exception:
            pass
        if not slim_terms:  # fallback: ancestor in slim
            ancestors = go_dag[go].get_all_parents()
            slim_terms.update([a for a in ancestors if a in goslim_dag])
    return list(slim_terms)

# Expand raw → slim
slim_map = []
for gid, df_sub in go_annot.groupby("gene_id"):
    go_ids = df_sub["annotation"].tolist()
    slim_terms = map_to_goslim(go_ids)
    for slim in slim_terms:
        slim_map.append({"gene_id": gid, "annotation": slim})
go_slim_df = pd.DataFrame(slim_map)
print(f"Annotated genes mapped to GO-slim: {go_slim_df['gene_id'].nunique()} / {go_annot['gene_id'].nunique()}")

# Run enrichment on raw + slim
for label, annotations in [("GO_raw", go_annot), ("GO_slim", go_slim_df)]:
    all_results = []
    for module in tqdm(corr_df['module'].unique(), desc=f"Modules ({label})"):
        sub = corr_df[corr_df['module'] == module].copy()
        sub = sub.sort_values(by="correlation", ascending=False)

        res = enrich_mwu(
            ordered_list=sub['gene_id'].values,
            annotation=annotations,
            min_size=MIN_SIZE,
            alternative="two-sided"
        )

        res['Module'] = module
        res['annotation_name'] = res['annotation'].map(go_id_to_name).fillna("NA")
        res = res[['Module','annotation','annotation_name','estimate','p_val','q_val','U','size','direction']]
        # Only keep annotations that are top enriched
        res = res[res['direction'] == 'top']
        # Save per-module results
        module_out = os.path.join(OUTDIR, f"{module}_{label}_MWU.csv")
        res.to_csv(module_out, index=False)

        sig = res[(res['q_val'] < QVAL_CUTOFF) & (res['estimate'] > EFFECT_CUTOFF)]

        if not sig.empty:
            all_results.append(sig)

    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        out_file = os.path.join(OUTDIR, f"MWU_significant_{label}.csv")
        combined.to_csv(out_file, index=False)
        print(f"Saved combined significant {label} results to {out_file}")
    else:
        print(f"No significant enrichments found for {label}")


=== Running GO enrichment (raw + slim) ===
Annotated genes for GO: 9764 / 32196
../data/annotation/functional_eggnog/go-basic.obo: fmt(1.2) rel(2024-10-27) 44,017 Terms; optional_attrs(relationship)
../data/annotation/functional_eggnog/goslim_generic.obo: fmt(1.2) rel(go/2024-09-08/subsets/goslim_generic.owl) 206 Terms
Annotated genes mapped to GO-slim: 6157 / 9764


Modules (GO_raw):  17%|█▋        | 1/6 [00:02<00:11,  2.31s/it]

Number of significant results: 71


Modules (GO_raw):  33%|███▎      | 2/6 [00:04<00:09,  2.36s/it]

Number of significant results: 63


...

Number of significant results: 8


Modules (GO_slim):  83%|████████▎ | 5/6 [00:04<00:00,  1.22it/s]

Number of significant results: 9


Modules (GO_slim): 100%|██████████| 6/6 [00:04<00:00,  1.22it/s]

Number of significant results: 3
Saved combined significant GO_slim results to ../data/analysis/WGCNA_130/enrichment/MWU_significant_GO_slim.csv


In [38]:
print("\n=== Running KOG enrichment (not just desc, but general categories) ===")

# Explode raw KOG
KOG_annot = explode_annotations(ann, "KOG")
print(f"Annotated genes for KOG: {KOG_annot['gene_id'].nunique()} / {ann['gene_id'].nunique()}")

# Load KOG mapping
kog_map = pd.read_csv("../data/phaglo1_mapping/annotation/kog_mapping_processed.tsv", sep="\t", index_col=False)

# Run enrichment
for label, annotations in [("KOG", KOG_annot)]:
    all_results = []
    for module in tqdm(corr_df['module'].unique(), desc=f"Modules ({label})"):
        sub = corr_df[corr_df['module'] == module].copy()
        sub = sub.sort_values(by="correlation", ascending=False)

        res = enrich_mwu(
            ordered_list=sub['gene_id'].values,
            annotation=annotations,
            min_size=MIN_SIZE,
            alternative="two-sided"
        )

        res['Module'] = module
        res = res.merge(kog_map, left_on='annotation', right_on='KOG', how='left')
        res['annotation_name'] = res['KOG_category'].fillna("NA")
        res = res[['Module','annotation','annotation_name','estimate','p_val','q_val','U','size','direction']]
        # Only keep annotations that are top enriched
        res = res[res['direction'] == 'top']
        # Save per-module results
        module_out = os.path.join(OUTDIR, f"{module}_{label}_MWU.csv")
        res.to_csv(module_out, index=False)

        sig = res[(res['q_val'] < QVAL_CUTOFF) & (res['estimate'] > EFFECT_CUTOFF)]

        if not sig.empty:
            all_results.append(sig)

    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        out_file = os.path.join(OUTDIR, f"MWU_significant_{label}.csv")
        combined.to_csv(out_file, index=False)
        print(f"Saved combined significant {label} results to {out_file}")
    else:
        print(f"No significant enrichments found for {label}")


=== Running KOG enrichment (not just desc, but general categories) ===
Annotated genes for KOG: 10068 / 32196


Modules (KOG):  17%|█▋        | 1/6 [00:00<00:03,  1.25it/s]

Number of significant results: 23


Modules (KOG):  33%|███▎      | 2/6 [00:01<00:03,  1.12it/s]

Number of significant results: 9


...

Number of significant results: 17


Modules (KOG):  83%|████████▎ | 5/6 [00:04<00:00,  1.13it/s]

Number of significant results: 21


Modules (KOG): 100%|██████████| 6/6 [00:05<00:00,  1.13it/s]

Number of significant results: 10
Saved combined significant KOG results to ../data/analysis/WGCNA_130/enrichment/MWU_significant_KOG.csv


In [ ]:
df = pd.read_csv(os.path.join(OUTDIR, "MWU_significant_GO_raw.csv"))

# Remove NA annotations
df = df[df['annotation_name'].notna()]

def get_top_go_terms_per_module(df, top_n=6):
    df['q_val'] = pd.to_numeric(df['q_val'], errors='coerce')
    df['estimate'] = pd.to_numeric(df['estimate'], errors='coerce')
    df['size'] = pd.to_numeric(df['size'], errors='coerce')
    df['score'] = -np.log10(df['q_val'] + 1e-10) * df['estimate'].abs() * df['size']
    top_go_terms = (
        df.sort_values("score", ascending=False)
          .groupby("Module")
          .head(top_n)
          .sort_values(["Module", "score"], ascending=[True, False])
    )
    return top_go_terms[[
        "Module", "annotation", "annotation_name",
        "q_val", "estimate", "size"
    ]]

get_top_go_terms_per_module(df)

,Module,annotation,annotation_name,q_val,estimate,size
2,black,GO:0008113,peptide-methionine (S)-S-oxide reductase activity,7.847087e-02,0.760053,9
3,black,GO:0006633,fatty acid biosynthetic process,7.975601e-02,0.757530,9
4,black,GO:0009654,photosystem II oxygen evolving complex,8.373035e-02,0.754029,9
0,black,GO:0004618,phosphoglycerate kinase activity,3.728884e-02,0.888157,5
1,black,GO:0004332,fructose-bisphosphate aldolase activity,7.644768e-02,0.796957,7
5,black,GO:0008324,monoatomic cation transmembrane transporter ac...,8.616455e-02,0.838207,5
46,blue,GO:0003676,nucleic acid binding,5.720359e-06,0.607529,224
49,blue,GO:0005515,protein binding,1.659403e-05,0.606762,202
48,blue,GO:0005634,nucleus,8.876741e-06,0.632849,137
45,blue,GO:0003677,DNA binding,5.720359e-06,0.643270,128


In [ ]:
# Export csv
get_top_go_terms_per_module(df).to_csv("../data/analysis/WGCNA_130/enrichment/top_GO_terms_per_module.csv", index=False)